# 03. Score V1 de priorização

Objetivo: ordenar a fila diária do comercial por chance de matrícula, no lugar da ordem de chegada (FIFO).

Premissas da V1 (prazo de 5 dias úteis):
- **Variáveis só com o que existe no momento de priorizar**: cadastro do lead e navegação até ele entrar na fila. Nada de status do CRM, data de matrícula ou páginas de pós-matrícula.
- **Score por pontos**: cada faixa de cada variável ganha pontos pela conversão histórica. Simples de explicar ao comercial e de auditar.
- **Base de aprendizado = leads que o FIFO já abordou**: como o FIFO escolhe por hora de chegada e não por intenção, esses leads são uma amostra sem viés de intenção, com o resultado de quem foi contatado.

Depende das tabelas do notebook 01 e da `tb_nav_sessao` do notebook 02.

In [0]:
%sql
USE cogna.captacao;

DECLARE OR REPLACE VARIABLE dt_fim DATE DEFAULT DATE'2026-11-22';
DECLARE OR REPLACE VARIABLE inicio_validacao DATE DEFAULT DATE'2026-09-01';
DECLARE OR REPLACE VARIABLE maturacao_dias INT DEFAULT 30;   -- tempo para o lead ter chance de converter
DECLARE OR REPLACE VARIABLE janela_nav_dias INT DEFAULT 14; -- navegação considerada antes do cadastro

In [0]:
# calendário de dias úteis (mesmo do notebook 02)
from datetime import date, timedelta

FERIADOS = ["2026-06-04", "2026-09-07", "2026-10-12", "2026-11-02", "2026-11-15", "2026-11-20"]
ini, fim = date(2026, 5, 1), date(2027, 1, 31)
dias = [ini + timedelta(days=i) for i in range((fim - ini).days + 1)]
linhas, prox = [], None
for d in reversed(dias):
    tipo = "feriado" if d.isoformat() in FERIADOS else ("fim_de_semana" if d.weekday() >= 5 else "dia_util")
    prox_depois = prox
    if tipo == "dia_util":
        prox = d
    linhas.append((d, tipo, prox, prox_depois))
spark.createDataFrame(linhas, "dt date, tipo_dia string, prox_dia_util date, prox_dia_util_depois date").createOrReplaceTempView("vw_calendario")

## 1. Variáveis por lead

Pensando na captação de ensino superior:
- **Curso e modalidade**: EAD e presencial têm jornadas diferentes; cursos de saúde e direito costumam ter decisão mais firme.
- **Canal** (nulo vira `desconhecido`): indicação e orgânico trazem intenção maior que mídia paga e afiliados.
- **Quando entrou**: período do dia e se foi em dia útil. Quem se cadastra à noite ou no fim de semana costuma estar pesquisando com calma.
- **Navegação**: sessões, tempo no site e, principalmente, visitas a preço, bolsas/descontos e inscrição, que são os sinais mais diretos de decisão.

Momento de corte: o lead entra na fila do dia útil (às 7h para quem chegou fora do expediente). Só a navegação até esse momento é usada.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_lead_base AS
WITH base AS (
  SELECT l.lead_id, l.dt_criacao, l.curso, l.modalidade, l.fl_matricula,
         coalesce(l.canal_origem, 'desconhecido') AS canal,
         CASE WHEN hour(l.dt_criacao) < 6 THEN 'madrugada'
              WHEN hour(l.dt_criacao) < 12 THEN 'manha'
              WHEN hour(l.dt_criacao) < 18 THEN 'tarde'
              ELSE 'noite' END AS periodo_entrada,
         CASE WHEN cal.tipo_dia = 'dia_util' THEN 'dia_util' ELSE 'fim_de_semana_feriado' END AS tipo_dia_entrada,
         CASE WHEN cal.tipo_dia <> 'dia_util' THEN cal.prox_dia_util
              WHEN hour(l.dt_criacao) >= 18 THEN cal.prox_dia_util_depois
              ELSE cal.dt END AS dia_fila
  FROM tb_leads l
  JOIN vw_calendario cal ON cal.dt = to_date(l.dt_criacao)
),
abordados AS (SELECT DISTINCT lead_id FROM tb_abordagem_crm)
SELECT b.*,
       greatest(b.dt_criacao, timestampadd(HOUR, 7, cast(b.dia_fila AS TIMESTAMP))) AS ts_corte,
       CASE WHEN a.lead_id IS NULL THEN 0 ELSE 1 END AS abordado
FROM base b
LEFT JOIN abordados a ON a.lead_id = b.lead_id

In [0]:
%sql
-- navegação por lead até o corte.
-- tb_nav_sessao liga a sessão inteira ao lead, inclusive os eventos anônimos antes do cadastro.
-- o filtro de data vem primeiro para ler só os arquivos do período (tabela clusterizada por data).
CREATE OR REPLACE TABLE tb_features_lead AS
WITH sessao_lead AS (
  SELECT session_id, lead_id FROM tb_nav_sessao WHERE lead_id IS NOT NULL
),
eventos AS (
  SELECT s.lead_id, n.session_id, n.ts_evento, n.pagina, n.tempo_permanencia_seg
  FROM tb_navegacao n
  JOIN sessao_lead s ON s.session_id = n.session_id
  WHERE n.ts_evento >= date_sub(dt_fim, 180 + janela_nav_dias)
    AND n.pagina NOT IN ('matricula_pagamento', 'area_do_aluno')
),
nav AS (
  SELECT b.lead_id,
         count(DISTINCT e.session_id) AS qtd_sessoes,
         count(e.session_id) AS qtd_eventos,
         coalesce(sum(e.tempo_permanencia_seg), 0) AS tempo_total_seg,
         max(CASE WHEN e.pagina = 'preco_mensalidade' THEN 1 ELSE 0 END) AS viu_preco,
         max(CASE WHEN e.pagina = 'bolsas_descontos' THEN 1 ELSE 0 END) AS viu_bolsas,
         max(CASE WHEN e.pagina = 'vestibular_inscricao' THEN 1 ELSE 0 END) AS viu_inscricao
  FROM vw_lead_base b
  LEFT JOIN eventos e
    ON e.lead_id = b.lead_id
   AND e.ts_evento <= b.ts_corte
   AND e.ts_evento >= b.dt_criacao - make_interval(0, 0, 0, janela_nav_dias)
  GROUP BY b.lead_id
)
SELECT b.lead_id, b.dt_criacao, b.dia_fila, b.ts_corte, b.abordado, b.fl_matricula,
       b.curso, b.modalidade, b.canal, b.periodo_entrada, b.tipo_dia_entrada,
       n.qtd_sessoes, n.qtd_eventos, n.tempo_total_seg, n.viu_preco, n.viu_bolsas, n.viu_inscricao
FROM vw_lead_base b
JOIN nav n ON n.lead_id = b.lead_id

In [0]:
%sql
SELECT * FROM tb_features_lead LIMIT 20

## 2. Score V1

Cada variável é colocada em faixas. Os pontos de cada faixa são o log da razão entre a conversão da faixa e a conversão média,
calculados nos leads abordados até agosto. A conversão da faixa é suavizada em direção à média para faixas pequenas não ganharem peso exagerado.
O score do lead é a soma dos pontos das suas faixas.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW vw_faixas AS
SELECT lead_id, dt_criacao, dia_fila, abordado, fl_matricula,
       stack(10,
         'curso', curso,
         'modalidade', modalidade,
         'canal', canal,
         'periodo_entrada', periodo_entrada,
         'tipo_dia_entrada', tipo_dia_entrada,
         'sessoes', CASE WHEN qtd_sessoes = 0 THEN '0' WHEN qtd_sessoes = 1 THEN '1' WHEN qtd_sessoes <= 3 THEN '2-3' ELSE '4+' END,
         'tempo_site', CASE WHEN tempo_total_seg = 0 THEN '0' WHEN tempo_total_seg < 120 THEN 'ate 2min'
                            WHEN tempo_total_seg < 600 THEN '2-10min' ELSE '10min+' END,
         'viu_preco', cast(viu_preco AS STRING),
         'viu_bolsas', cast(viu_bolsas AS STRING),
         'viu_inscricao', cast(viu_inscricao AS STRING)
       ) AS (variavel, faixa)
FROM tb_features_lead

In [0]:
%sql
CREATE OR REPLACE TABLE tb_pesos_score AS
WITH treino AS (
  SELECT * FROM vw_faixas WHERE abordado = 1 AND dt_criacao < inicio_validacao
),
media AS (
  SELECT avg(fl_matricula) AS taxa_media FROM treino WHERE variavel = 'curso'
)
SELECT t.variavel, t.faixa,
       count(*) AS leads,
       sum(t.fl_matricula) AS matriculas,
       round(100 * avg(t.fl_matricula), 2) AS pct_matricula,
       round(ln(((sum(t.fl_matricula) + 50 * m.taxa_media) / (count(*) + 50)) / m.taxa_media), 3) AS pontos
FROM treino t CROSS JOIN media m
GROUP BY t.variavel, t.faixa, m.taxa_media

In [0]:
%sql
-- o que mais pesa no score
SELECT variavel, faixa, leads, pct_matricula, pontos
FROM tb_pesos_score
ORDER BY variavel, pontos DESC

In [0]:
%sql
CREATE OR REPLACE TABLE tb_score_lead AS
SELECT f.lead_id, f.dia_fila,
       round(sum(coalesce(p.pontos, 0)), 3) AS score
FROM vw_faixas f
LEFT JOIN tb_pesos_score p ON p.variavel = f.variavel AND p.faixa = f.faixa
GROUP BY f.lead_id, f.dia_fila

In [0]:
%sql
-- exemplo da lista que o comercial receberia: fila de um dia útil ordenada pelo score
SELECT s.dia_fila, s.lead_id, s.score, f.curso, f.modalidade, f.canal, f.periodo_entrada,
       f.qtd_sessoes, f.viu_preco, f.viu_bolsas, f.viu_inscricao
FROM tb_score_lead s
JOIN tb_features_lead f ON f.lead_id = s.lead_id
WHERE s.dia_fila = (SELECT max(dia_fila) FROM tb_score_lead WHERE dia_fila <= dt_fim)
ORDER BY s.score DESC
LIMIT 50

## 3. Por que o score avança sobre o FIFO

Não dá para simular o resultado de quem não foi abordado: só sabemos a conversão de quem o comercial contatou.
Mas o FIFO escolhe por hora de chegada, que não tem relação com intenção, então os leads que ele abordou funcionam como uma amostra aleatória do ponto de vista do score.

Nesses leads, de setembro em diante (período que não entrou no cálculo dos pontos) e só com quem já teve 30 dias para converter, separamos em 5 faixas de score.
Se o score ordena bem, a faixa de cima converte bem acima da média, e a média é exatamente o que o FIFO entrega hoje.
Com capacidade de 20%, a V1 passaria a abordar a faixa de cima.

In [0]:
%sql
WITH validacao AS (
  SELECT f.lead_id, f.fl_matricula, s.score
  FROM tb_features_lead f
  JOIN tb_score_lead s ON s.lead_id = f.lead_id
  WHERE f.abordado = 1
    AND f.dt_criacao >= inicio_validacao
    AND f.dt_criacao < date_sub(dt_fim, maturacao_dias)
),
faixas AS (
  SELECT *, ntile(5) OVER (ORDER BY score DESC) AS faixa_score FROM validacao
)
SELECT faixa_score,
       count(*) AS leads,
       sum(fl_matricula) AS matriculas,
       round(100 * avg(fl_matricula), 2) AS pct_matricula,
       round(100 * sum(fl_matricula) / sum(sum(fl_matricula)) OVER (), 1) AS pct_das_matriculas
FROM faixas
GROUP BY faixa_score
ORDER BY faixa_score

In [0]:
%sql
-- resumo: FIFO (média dos abordados) x 20% de maior score, no mesmo período
WITH validacao AS (
  SELECT f.fl_matricula, s.score
  FROM tb_features_lead f
  JOIN tb_score_lead s ON s.lead_id = f.lead_id
  WHERE f.abordado = 1
    AND f.dt_criacao >= inicio_validacao
    AND f.dt_criacao < date_sub(dt_fim, maturacao_dias)
),
corte AS (SELECT percentile(score, 0.8) AS p80 FROM validacao)
SELECT round(100 * avg(fl_matricula), 2) AS pct_matricula_fifo,
       round(100 * avg(CASE WHEN score >= p80 THEN fl_matricula END), 2) AS pct_matricula_top20_score,
       round(avg(CASE WHEN score >= p80 THEN fl_matricula END) / avg(fl_matricula), 2) AS ganho_vezes
FROM validacao CROSS JOIN corte

**Como ler**: `ganho_vezes` é quantas vezes a faixa de maior score converte mais que o FIFO.
É uma estimativa feita em leads que já foram abordados; o ganho real se confirma colocando a V1 no ar com parte da capacidade ainda no FIFO e comparando as duas listas.

**Limites da V1**
- Variáveis correlacionadas (sessões e tempo no site, por exemplo) somam pontos parecidos; uma regressão logística resolve isso na V2.
- Navegação anterior ao cadastro em outras sessões continua anônima; só a sessão do cadastro é ligada ao lead.
- O score mede quem converte quando é contatado, não quanto o contato muda a decisão. Medir esse efeito exige o teste com parte da capacidade no FIFO.

## Conclusão

Na parte de dados, tratei as três tabelas pensando no que realmente existe na hora de priorizar. O canal nulo virou `desconhecido`, porque na maioria dos casos ele indica tráfego sem utm, e isso também é informação. A navegação anônima da sessão de cadastro foi ligada ao lead pelo `session_id`, e deixei de fora o status do CRM, a data de matrícula e as páginas de pós matrícula, que entregariam a resposta. Como a navegação tem bilhões de linhas, a leitura é sempre por período e a classificação das sessões é incremental.

Para o score, escolhi uma soma de pontos com variáveis de cadastro e de navegação, aprendida nos leads que o FIFO já abordou. Cabe no prazo, qualquer pessoa do comercial entende por que um lead está no topo da lista e cada peso pode ser conferido na `tb_pesos_score`.

O ganho sobre o FIFO vem de trocar a ordem de chegada, que não tem relação com intenção, por sinais de interesse do lead. Nos leads de validação, a faixa de maior score converte bem acima da média, que é o que o FIFO entrega hoje. Abordar esses 20% no lugar dos primeiros que chegaram traz mais matrículas com a mesma equipe.

Para entrar no ar em 5 dias, os dois primeiros ficam para validar variáveis e score com o comercial, o terceiro para publicar a lista priorizada no CRM e os dois últimos para colocar em produção mantendo parte da capacidade no FIFO, o que permite medir o ganho real.

Na V2, a soma de pontos pode ser substituída por uma regressão logística, com mais sinais de navegação, score recalculado ao longo do dia e um modelo de uplift para priorizar os leads em que o contato faz mais diferença.
